In [1]:
import pandas as pd
import os
import pickle 
import numpy as np

In [2]:
import scanpy as sc
split = 5
adata_train_path = f"/lustre/groups/ml01/workspace/ot_perturbation/data/sciplex/adata_train_{split}.h5ad"
adata_ood_path = f"/lustre/groups/ml01/workspace/ot_perturbation/data/sciplex/adata_ood_{split}.h5ad"
adata_train_sci = sc.read_h5ad(adata_train_path)
adata_ood_sci = sc.read_h5ad(adata_ood_path)
adata_train_obs = adata_train_sci.obs.drop_duplicates(subset="condition")
adata_ood_obs = adata_ood_sci.obs.drop_duplicates(subset="condition")
sciplex_obs = pd.concat((adata_train_obs, adata_ood_obs))

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [3]:
sciplex_obs2 = sciplex_obs[sciplex_obs["drug"]!="Vehicle"]
sciplex_obs2["drug_cell_line"] = sciplex_obs2["cell_line"].astype("str") + "_" + sciplex_obs2["drug"].astype("str")
drug_cell_line_combinations_sciplex = set(sciplex_obs2["drug_cell_line"].values)
len(drug_cell_line_combinations_sciplex)

/tmp/ipykernel_21711/3002172320.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sciplex_obs2["drug_cell_line"] = sciplex_obs2["cell_line"].astype("str") + "_" + sciplex_obs2["drug"].astype("str")


560

In [4]:
model = "charmed-deluge-4206_CellFlow.pkl"
input_dir = '/lustre/groups/ml01/workspace/ot_perturbation/models/cellflow/tahoe'
wandb_name = model.split("_")[0]

In [5]:
df_sci_emb = pd.read_csv(os.path.join(input_dir, f"condition_embeddings_sciplex_{wandb_name}.csv"))

In [6]:
df_sci_emb["cell_line"] = df_sci_emb.apply(lambda x: x["condition"].split("_")[0], axis=1)
df_sci_emb["dose"] = df_sci_emb.apply(lambda x: x["condition"].split("_")[-1], axis=1)
df_sci_emb["drug"] = df_sci_emb.apply(lambda x: "_".join(x["condition"].split("_")[1:-1]), axis=1)
df_sci_emb["cell_line_drug"] = df_sci_emb["cell_line"] + "_" + df_sci_emb["drug"]

dict_sci_emb = dict(zip(df_sci_emb["cell_line_drug"], np.array(df_sci_emb[[str(el) for el in np.arange(128)]].values.tolist())))


In [7]:
drug_cell_line_combinations_sciplex - set(dict_sci_emb.keys())

set()

In [8]:
# (drug, cell line)-combined embeddings obtained from drug embeddings, cell line embeddings, and highest TAHOE dosage
out_dir = "/lustre/groups/ml01/workspace/ot_perturbation/models/cellflow/embeddings_from_tahoe"
with open(os.path.join(out_dir, "sci_emb.pkl"), "wb") as f:
    pickle.dump(dict_sci_emb, f)

In [9]:
df_tahoe = pd.read_csv(os.path.join(input_dir, f"sci_drug_with_tahoe_cl_{wandb_name}.csv"))

In [10]:
df_tahoe["cell_line"] = df_tahoe.apply(lambda x: "_".join(x["condition"].split("_")[-2:]), axis=1)
df_tahoe["dose"] = df_tahoe.apply(lambda x: x["condition"].split("_")[-3], axis=1)
df_tahoe["drug"] = df_tahoe.apply(lambda x: "_".join(x["condition"].split("_")[:-3]), axis=1)


In [11]:
concat_dict = {}
for drug in df_tahoe["drug"].unique():
    df_tahoe_red = df_tahoe[df_tahoe["drug"]==drug]
    embs = []
    for cell_line in df_tahoe["cell_line"].unique():
        embs.append(np.array(df_tahoe_red[df_tahoe_red["cell_line"]==cell_line][[str(el) for el in np.arange(128)]].values.tolist()))
    concat_dict[drug] = np.squeeze(np.concatenate(embs, axis=1))

In [12]:
set(sciplex_obs["drug"].unique()) - set(concat_dict.keys())

{'Vehicle'}

In [13]:
with open(os.path.join(out_dir, "sci_drug_concatenated_from_tahoe.pkl"), "wb") as f:
    pickle.dump(concat_dict, f)

In [14]:
df_sci_cl_cell_line = pd.read_csv(os.path.join(input_dir, f"condition_embeddings_sciplex_closest_cell_line_{wandb_name}.csv"))

df_sci_cl_cell_line["cell_line"] = df_sci_cl_cell_line.apply(lambda x: x["condition"].split("_")[0], axis=1)
df_sci_cl_cell_line["dose"] = df_sci_cl_cell_line.apply(lambda x: x["condition"].split("_")[-1], axis=1)
df_sci_cl_cell_line["drug"] = df_sci_cl_cell_line.apply(lambda x: "_".join(x["condition"].split("_")[1:-1]), axis=1)
df_sci_cl_cell_line["cell_line_drug"] = df_sci_cl_cell_line["cell_line"] + "_" + df_sci_cl_cell_line["drug"]

dict_sci_cl_cell_line = dict(zip(df_sci_cl_cell_line["cell_line_drug"], np.array(df_sci_cl_cell_line[[str(el) for el in np.arange(128)]].values.tolist())))


In [15]:
# (drug, cell line)-combined embeddings obtained from drug embeddings, closest cell line embeddings observed in TAHOE, and highest TAHOE dosage
out_dir = "/lustre/groups/ml01/workspace/ot_perturbation/models/cellflow/embeddings_from_tahoe"
with open(os.path.join(out_dir, "sci_cl_cell_line.pkl"), "wb") as f:
    pickle.dump(dict_sci_cl_cell_line, f)

In [16]:
drug_cell_line_combinations_sciplex - set(dict_sci_cl_cell_line.keys())

set()

In [17]:
dict_sci_cl_cell_line['K562_(+)-JQ1']

array([ 0.07979356,  0.09428334, -0.00250581,  0.10420562, -0.04957916,
       -0.04073947,  0.03237542, -0.03359822,  0.05177935,  0.12841418,
        0.09159313,  0.10584505, -0.07147624, -0.05772048, -0.07663977,
        0.06537401, -0.00548801, -0.00746838, -0.08064277, -0.03957898,
        0.01286256, -0.02795728,  0.05949775,  0.05837837,  0.04437872,
       -0.06726164,  0.07747285, -0.09217363,  0.04946553, -0.09405426,
       -0.08537797,  0.01233848, -0.11114095,  0.02120531,  0.08675147,
       -0.03021088, -0.02691852,  0.02583191,  0.07060506,  0.02107217,
        0.05478718, -0.10991467,  0.02498225, -0.0514869 ,  0.0332434 ,
        0.01847397, -0.04467416,  0.03362626,  0.063887  , -0.08282245,
        0.07726684,  0.03508331,  0.04696121, -0.06184305, -0.04805879,
        0.01361058,  0.04540355,  0.10172075, -0.02053569,  0.06015344,
       -0.08480794,  0.02165711,  0.09164091, -0.0712859 , -0.07317873,
        0.04882861, -0.01631773,  0.0678424 ,  0.00975217, -0.08

In [18]:
dict_sci_emb['K562_(+)-JQ1']

array([ 0.05709351, -0.1313168 ,  0.23918954,  0.00878576, -0.08376697,
       -0.09273828, -0.41108626,  0.14920539,  0.0302663 ,  0.03535541,
        0.11784637, -0.270637  , -0.21109454, -0.36169368,  0.02505426,
        0.2002581 ,  0.23474877, -0.1012875 ,  0.02678138, -0.27860963,
        0.27587628, -0.13723995,  0.04353546, -0.10167287,  0.21012887,
       -0.18883994, -0.09612263,  0.07900716, -0.3826259 , -0.36321798,
        0.29087332,  0.37003717,  0.3537568 , -0.09595973, -0.20167387,
       -0.00354319, -0.0265218 ,  0.25130075,  0.39733014,  0.34573138,
        0.20803337, -0.39034262, -0.14537267, -0.11788032, -0.6960582 ,
        0.23163429,  0.17797154,  0.1270513 , -0.15609008,  0.02593284,
        0.08649333,  0.3432415 , -0.4295548 , -0.14842401,  0.13425781,
       -0.2789721 ,  0.19764689,  0.0907098 , -0.2971194 ,  0.03999401,
       -0.39154875,  0.41758323,  0.11119852, -0.17593017,  0.05338104,
       -0.15836318, -0.42109346,  0.07011143,  0.48958793, -0.15